In [1]:
from langchain_anthropic import ChatAnthropic
from dotenv import load_dotenv
import os
 
load_dotenv(override=True)
 
api_key = os.getenv("ANTHROPIC_API_KEY")
if api_key is None:
    raise ValueError("ANTHROPIC_API_KEY not found in .env file")
base_url=os.getenv("BASE_URL")
model_name=os.getenv("MODEL_NAME")
 
llm = ChatAnthropic(model=model_name,base_url=base_url,anthropic_api_key=api_key, max_tokens=200)

prompt = ChatPromptTemplate.from_template('Tell me a fun fact about {topic}.')
 
chain = prompt | llm          # prompt --> llm
 
response = chain.invoke({'topic': 'space'})
print(type(response).__name__)    # AIMessage
print(response. Content)

NameError: name 'ChatPromptTemplate' is not defined

In [ ]:
chain = prompt | llm | StrOutputParser()   # prompt --> llm --> parser
 
response = chain.invoke({'topic': 'black holes'})
print(type(response))    # <class 'str'>  (plain string, not AIMessage)
print(response)

In [ ]:
# Step 1: summarise a topic

summary_prompt = ChatPromptTemplate.from_template('Summarize {topic} in 2 sentences.')

summary_chain  = summary_prompt | llm | StrOutputParser()
 
# Step 2: turn that summary into a tweet

tweet_prompt   = ChatPromptTemplate.from_template('Turn this into a tweet:\n{summary}')

tweet_chain    = tweet_prompt | llm | StrOutputParser()
 
# Connect: summary output --> tweet input

full_chain = summary_chain | (lambda s: {'summary': setattr}) | tweet_chain
 
result = full_chain.invoke({'topic': 'Quantum Computing'})

print(result)
 
from langchain_core.runnables import RunnableParallel
 
pros_prompt = ChatPromptTemplate.from_template('List 2 pros of {technology}.')

cons_prompt = ChatPromptTemplate.from_template('List 2 cons of {technology}.')
 
# Both chains run IN PARALLEL

parallel_chain = RunnableParallel(

    pros_chain = pros_prompt | llm | StrOutputParser(),

    cons_chain = cons_prompt | llm | StrOutputParser(),

)
 
result = parallel_chain.invoke({'technology': 'AI'})

print('PROS:\n', result['pros_chain'])

print()

print('CONS:\n', result['cons_chain'])
 